# nb_05a — Signed-laterality decodability probe (E1, hardened)

**Does a self-supervised skeleton world model learn that the body is bilaterally symmetric?** This notebook makes the question exact, audits a frozen **S-JEPA** against it, and finds a *robust informative null*: the symmetry does **not** emerge.

This is experiment **E1** of the NeurIPS 2026 *Physical World AI* laterality package (`../neurips-laterality/docs/`). It **binds to the canonical checkpoint** `sjepa_curriculum_final.pt` (fingerprint `7d13841a…`) and hardens the original probe with: SVD-solver ridge, **repeated** source-disjoint CV (10 reshuffles → stability interval), an **alpha-sensitivity sweep**, an explicit **cohort decision** (626 modeled / 642 superset), and a **landmark-missingness control lane**.

> **Responsible-use notice.** All results in this notebook are **transductive** — the
> S-JEPA encoder was trained on the very sequences being evaluated — so they carry
> *internal validity only* and make **no** claim of generalization to new sources or
> people. The **source video** (not the clip, not the individual) is the independent unit
> of analysis. The dataset's condition folders (`normal`, `parkinsons`, `stroke`,
> `myopathic`, `cerebralpalsy`) are **dataset annotations, not diagnoses**. The dataset's
> official distribution provides annotations and public video URLs, not raw video; this
> analysis uses derived pose sequences, infers no identity, and redistributes no raw or
> identity-bearing frames. **No institutional ethics determination or completed data-use
> review is yet on record; both must be resolved before submission.**

## 1. Bilateral symmetry as a $\mathbb{Z}/2$ group action

Reflecting a skeleton across the sagittal plane **and** swapping left/right landmarks yields another valid skeleton. The mirror operator $M$ negates the $x$-coordinate of every joint and swaps **all sixteen** bilateral landmark pairs — a valid whole-body reflection. Then $M^2=I$, so $G=\{I,M\}\cong\mathbb{Z}/2$.

**Signed laterality target.** Over the six bilateral pairs that carry gait laterality

$$(11,12),\ (23,24),\ (25,26),\ (27,28),\ (29,30),\ (31,32)$$

(shoulders, hips, knees, ankles, heels, foot-indices), let $\ell_k$ (resp. $r_k$) be the per-joint *temporal* standard deviation of the left (resp. right) landmark, summed over $x,y,z$. Define

$$ y(x) \;=\; \sum_k (\ell_k - r_k). $$

Mirroring swaps $\ell_k \leftrightarrow r_k$, so **by construction**

$$ y(Mx) = -\,y(x), \qquad\text{i.e.}\qquad y(T_g\,x)=\rho(g)\,y(x),\quad \rho(I)=+1,\ \rho(M)=-1. $$

A world model *"encodes bilateral symmetry"* iff a read-out of its features reproduces this antisymmetry — a **mirror slope of $-1$** — while remaining **decodable** ($R^2$ well above an untrained floor). This is a *geometry-aware* probe of an *articulated, deformable* body whose target is a *proprioceptive* quantity (which side moves more) — the workshop's themes made concrete. See `../neurips-laterality/docs/figures/fig1_group_action.svg`.

## 2. The audit protocol

**Five lanes** (`../neurips-laterality/docs/figures/fig2_audit_protocol.svg`):

| Lane | Feature | Role |
|------|---------|------|
| **A — learned** | per-pair $[\ell-r,\ \ell+r]$ token statistics from the frozen encoder | the thing under test |
| **B — raw ceiling** | target regressed on raw coordinates ($R^2\approx1$) | decodability sanity check |
| **C — floor** | identical architecture, **random** weights | untrained baseline |
| **D — pooled** | whole-body mean/std, side-blind | must stay low |
| **E — missingness** | per-joint left/right valid-fraction | gauges how much of the target pure left/right *visibility* explains |

**Estimator.** Source-video-disjoint `GroupKFold` on `video_id`; inner ridge-penalty selection over $\alpha\in\mathrm{logspace}(-3,3,13)$; per-fold standardization; **SVD** solver (removes the ill-conditioning of the original run). We report **repeated** shuffled grouped CV over **10 reshuffles** as mean $\pm$ a $t$-based **stability interval** ($t^\*=2.262$, $\mathrm{df}=9$). This interval measures sensitivity to the *partition* under the fixed 626/93 cohort — not population sampling — so interval overlap is a stability heuristic, **not** a significance test.

**Pre-registered gates.** (i) A beats C by $\ge 0.05\ R^2$; (ii) A reaches $\ge 80\%$ of B; (iii) sign correct on $\ge 75\%$ of held-out sources. **Secondary geometry band:** mirror slope negative and within $[-1.25,-0.80]$.

**Cohort decision.** The checkpoint's `sequence_ids` are exactly **626** sequences / **93** source videos — that is the **PRIMARY** (fully transductive) cohort. The **642** pose-available superset (adds 16 coverage-QC-dropped rows the encoder never saw) is a **robustness** view only, and it reproduces the pre-hardening canonical numbers bit-for-bit (a pipeline check).

In [1]:
import json, os, subprocess, sys, textwrap
from pathlib import Path

def find_experiment_dir(start=None):
    candidates = []
    if os.getenv("ALEXPOSE_ROOT"):
        env_root = Path(os.environ["ALEXPOSE_ROOT"]).expanduser().resolve()
        candidates.extend([env_root, env_root / "experiments" / "sjepa" / "gavd5-drift"])
    start = Path(start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        candidates.extend([base, base / "experiments" / "sjepa" / "gavd5-drift"])
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "work" / "experiments").is_dir():
            return candidate
    raise FileNotFoundError(f"Cannot locate gavd5-drift from {start}; set ALEXPOSE_ROOT.")


EXPERIMENT_DIR = find_experiment_dir()
NOTEBOOK_DIR = EXPERIMENT_DIR / "neurips-brain-body"
PY = EXPERIMENT_DIR / ".venv" / "bin" / "python"
PY = str(PY if PY.exists() else sys.executable)   # fall back to the running kernel
ART = EXPERIMENT_DIR / "work" / "artifacts" / "real"

def run_experiment(script_relpath):
    """Run a validated standalone experiment script; it regenerates its JSON artifact."""
    script = EXPERIMENT_DIR / script_relpath
    print(f"running {script.name} with {PY} ...")
    proc = subprocess.run([PY, str(script)], cwd=str(EXPERIMENT_DIR),
                          capture_output=True, text=True)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print("STDERR (tail):\n", proc.stderr[-4000:])
        raise RuntimeError(f"{script.name} exited {proc.returncode}")
    return proc

def approx(a, b, tol):
    return abs(float(a) - float(b)) <= tol


In [2]:
# Re-run the hardened E1 probe end-to-end (regenerates the canonical JSON).
run_experiment('work/experiments/e1_laterality_hardened.py')
res = json.loads((ART / 'idea5_signed_laterality_result_hardened.json').read_text())
prim = res['primary_cohort']; rob = res['robustness_cohort']
print('fingerprint', res['fingerprint'][:12], '| primary',
      prim['n_sequences'], 'seq /', prim['n_sources'], 'sources')

running e1_laterality_hardened.py with /Users/pmui/dev/alexpose/experiments/sjepa/gavd5-drift/.venv/bin/python ...


checkpoint 7d13841aceac  config={'frames': 64, 'joints': 33, 'coordinate_dim': 3, 'segment_length': 4, 'embed_dim': 96, 'encoder_depth': 4, 'predictor_depth': 2, 'heads': 4}
train sequence_ids: 626  | FRAMES=64 EMBED_DIM=96 SEGMENTS=16
642 availability: 642 seq / 94 sources
626 modeled     : 626 seq / 93 sources

=== PRIMARY: 626 modeled cohort (fully transductive) ===
{
  "lanes": {
    "A_learned": {
      "r2": 0.26839426283742185,
      "mae": 2.037501857331948
    },
    "B_raw_null": {
      "r2": 0.9999999999968954,
      "mae": 3.545913241042281e-06
    },
    "C_floor": {
      "r2": 0.22868171459099496,
      "mae": 2.032425014890784
    },
    "D_pooled": {
      "r2": 0.10666202348535614,
      "mae": 2.1664077394548613
    },
    "E_missingness": {
      "r2": 0.16229457719012086,
      "mae": 2.0024851705548676
    }
  },
  "mirror": {
    "slope": -0.7034917702207186,
    "flips": false
  },
  "sign_consistency": 0.5483870967741935,
  "missingness_r2": 0.1622945771901208

In [3]:
# ---- Table 1: five lanes on the 626 PRIMARY cohort (single-partition + repeated-CV CI)
ci = prim['repeated_cv_ci95']
def row(name, key_single, key_ci):
    s = prim['lanes'][key_single]['r2']
    if key_ci and key_ci in ci:
        d = ci[key_ci]; band = f"{d['mean']:.3f} [{d['ci95_lo']:.3f}, {d['ci95_hi']:.3f}]"
    else:
        band = '—'
    print(f"  {name:26s} single={s:+.3f}   repeated-CV={band}")
print('E1 laterality probe — 626 primary (canonical checkpoint)')
row('A — learned',           'A_learned',     'A_learned')
row('C — untrained floor',   'C_floor',       'C_floor')
row('E — missingness-only',  'E_missingness', 'E_missingness')
row('D — pooled (side-blind)','D_pooled',     'D_pooled')
row('B — raw ceiling',       'B_raw_null',    None)
print()
print(f"  mirror slope           {prim['mirror']['slope']:+.4f}  (flips={prim['mirror']['flips']})")
asc = ci['A_sign_consistency']
print(f"  sign consistency (A)   {asc['mean']:.3f} [{asc['ci95_lo']:.3f}, {asc['ci95_hi']:.3f}]")
print(f"  gates: beats_floor={prim['beats_floor_by_0.05']}  reaches_80pct_B={prim['reaches_80pct_of_null']}  "
      f"sign_ok={prim['sign_consistent_75pct']}")
print('  PRIMARY_VERDICT:', prim['PRIMARY_VERDICT'])

E1 laterality probe — 626 primary (canonical checkpoint)
  A — learned                single=+0.268   repeated-CV=0.198 [0.175, 0.221]
  C — untrained floor        single=+0.229   repeated-CV=0.245 [0.214, 0.275]
  E — missingness-only       single=+0.162   repeated-CV=0.202 [0.173, 0.231]
  D — pooled (side-blind)    single=+0.107   repeated-CV=0.101 [0.089, 0.114]
  B — raw ceiling            single=+1.000   repeated-CV=—

  mirror slope           -0.7035  (flips=False)
  sign consistency (A)   0.576 [0.548, 0.604]
  gates: beats_floor=False  reaches_80pct_B=False  sign_ok=False
  PRIMARY_VERDICT: INFORMATIVE NULL


In [4]:
# ---- Alpha-sensitivity sweep for Lane A (why the feature is weak/collinear)
sweep = prim['alpha_sweep_A']
print('alpha        R2(A)')
for a, r2 in sweep.items():
    print(f'  {a:>8s}  {r2:+.3f}')
lo = min(float(v) for v in sweep.values()); hi = max(float(v) for v in sweep.values())
print(f'\n  A decodes only under heavy regularization: {lo:+.2f} -> {hi:+.2f}')

alpha        R2(A)
     0.001  -2.036
  0.003162  -2.017
      0.01  -1.960
   0.03162  -1.803
       0.1  -1.457
    0.3162  -0.953
         1  -0.495
     3.162  -0.177
        10  +0.029
     31.62  +0.152
       100  +0.228
     316.2  +0.268
      1000  +0.268

  A decodes only under heavy regularization: -2.04 -> +0.27


In [5]:
# ---- Robustness cohort (642) reproduces the pre-hardening canonical numbers
print('642 robustness:',
      f"A={rob['lanes']['A_learned']['r2']:.3f}  C={rob['lanes']['C_floor']['r2']:.3f}  "
      f"slope={rob['mirror']['slope']:+.3f}  verdict={rob['PRIMARY_VERDICT']}")
print('  (single-partition A-C =',
      f"{rob['lanes']['A_learned']['r2'] - rob['lanes']['C_floor']['r2']:+.3f}; "
      'the lone gate-pass that does NOT survive cohort-matching + repeated CV)')

642 robustness: A=0.241  C=0.190  slope=-0.627  verdict=INFORMATIVE NULL
  (single-partition A-C = +0.051; the lone gate-pass that does NOT survive cohort-matching + repeated CV)


In [6]:
# ---- Assert the paper's E1 numbers trace to this freshly-written artifact
A = ci['A_learned']; C = ci['C_floor']; E = ci['E_missingness']; D = ci['D_pooled']
assert approx(A['mean'], 0.198, 0.01), A
assert approx(C['mean'], 0.245, 0.01), C
assert approx(E['mean'], 0.202, 0.02), E
assert approx(D['mean'], 0.101, 0.01), D
assert approx(prim['mirror']['slope'], -0.703, 0.02), prim['mirror']['slope']
assert A['mean'] < C['mean'], 'learned should NOT beat floor under repeated CV'
assert not (prim['beats_floor_by_0.05'] and prim['reaches_80pct_of_null']
            and prim['sign_consistent_75pct']), 'all three gates must fail'
print('OK — E1 numbers match the paper; robust informative null confirmed.')

OK — E1 numbers match the paper; robust informative null confirmed.


## 3. Reading the result

The learned lateral feature reaches only $R^2\approx0.198$ under repeated CV — **below the untrained floor** $\approx0.245$ **in mean**, with overlapping stability intervals (a partition-stability heuristic, not a significance test) — and a mirror slope of $\approx-0.70$ instead of the required $-1$. **All three pre-registered gates fail.** The single-partition ordering $A>C$ *reverses* under repetition (the original notebook's lone "win" was a single partition near the top of A's spread), and the alpha sweep shows A decodes only under heavy regularization — consistent with a weak, collinear feature rather than a clean axis. The missingness lane $\approx0.202$ has a mean close to A ($\approx0.198$), flagging possible visibility confounding and corroborating the null.

**Verdict: a robust informative null** — the world model did *not* learn bilateral symmetry as a decodable, sign-flipping axis. See `../neurips-laterality/docs/figures/fig3_e1_null.svg`. Next we ask whether the geometry can be **recovered by construction** — `nb_05c` (read-out) and `nb_05d` (encoder).